# HyperCube Multimodal Quickstart

이 컨테이너는 시작될 때 다음을 자동으로 했어요:

1. 마운트된 `*.tar.gz` 모델 아카이브를 `/workspace/<asset-slug>/`에 추출
2. 알려진 모델(`qwen2-vl-2b-instruct`)이면 백그라운드로 transformers 로드 + gradio UI(:7860) 자동 실행
3. `jupyter-server-proxy` 가 그 gradio를 현재 워크스페이스 URL 안의 `/proxy/7860/` 로 노출

아래 셀을 한 번 실행하면 gradio UI가 노트북 출력 영역에 inline iframe으로 뜹니다. 이미지를 업로드해서 바로 써보세요. 모델이 로드되는 동안에는 "503 / loading" 가 잠깐 뜰 수 있어요.

In [ ]:
import time, urllib.request, urllib.error
from IPython.display import IFrame, display, Markdown

GRADIO_URL = 'proxy/7860/'

for attempt in range(60):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:7860/', timeout=1) as r:
            if r.status < 500:
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    if attempt == 0:
        display(Markdown('Qwen2-VL 모델을 로딩 중입니다 (보통 30~60초)...'))
    time.sleep(2)

display(Markdown('### Gradio UI'))
IFrame(src=GRADIO_URL, width='100%', height=720)

## 직접 Python에서 호출하고 싶다면

백그라운드 gradio가 이미 모델을 로딩했더라도, 노트북에서 별도로 transformers를 호출할 수도 있습니다. 한 번 더 로드되어 메모리는 더 먹게 됩니다.

In [ ]:
# import torch
# from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
# from PIL import Image
# MODEL = '/workspace/qwen2-vl-2b-instruct'
# processor = AutoProcessor.from_pretrained(MODEL)
# model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='cuda')
# img = Image.open('your_image.png')
# msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': '설명해줘'}]}]
# text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
# inputs = processor(text=[text], images=[img], return_tensors='pt').to('cuda')
# out = model.generate(**inputs, max_new_tokens=256)
# print(processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0])